# MT-SICS Backend Hardware Validation

Run this notebook connected to a physical Mettler Toledo scale to validate
every implemented backend method. Each cell tests one method and logs the
raw MT-SICS command/response.

**Requirements:**
- Physical Mettler Toledo scale connected via USB-to-serial
- Scale powered on and warmed up (60 min)
- Weighing pan empty at start

---
## 1. Setup and Logging

In [ ]:
import logging
from datetime import datetime

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO
from pylabrobot.scales import Scale
from pylabrobot.scales.mettler_toledo import MettlerToledoWXS205SDUBackend

timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
log_file = f"./logs/hardware_validation/{timestamp}_validation.log"

pylabrobot.verbose(True, level=LOG_LEVEL_IO)
pylabrobot.setup_logger(log_dir=log_file, level=LOG_LEVEL_IO)

plr_logger = logging.getLogger("pylabrobot")
plr_logger.info("=== Hardware validation started ===")

In [ ]:
# Update port for your system
backend = MettlerToledoWXS205SDUBackend(port="/dev/ttyUSB2")
scale = Scale(name="validation_scale", backend=backend, size_x=0, size_y=0, size_z=0)

await scale.setup()

### Verify setup populated device identity

In [ ]:
print(f"Device type:    {backend.device_type}")
print(f"Serial number:  {backend.serial_number}")
print(f"Capacity:       {backend.capacity} g")
print(f"MT-SICS levels: {sorted(backend._mt_sics_levels)}")

assert backend.device_type is not None
assert backend.serial_number is not None
assert backend.capacity > 0
assert 0 in backend._mt_sics_levels
print("PASS: setup populated all identity fields")

---
## 2. Level 0 Commands (Basic Set)

### @ - Cancel (reset to determined state)

In [ ]:
serial_number = await backend.cancel()
print(f"Cancel returned serial number: {serial_number}")
assert serial_number == backend.serial_number
print("PASS: cancel returns correct serial number")

### I4 - Serial number

In [ ]:
sn = await backend.request_serial_number()
print(f"Serial number: {sn}")
assert len(sn) > 0
print("PASS: serial number is non-empty")

### I2 - Device type and capacity

In [ ]:
device_type = await backend.request_device_type()
capacity = await backend.request_capacity()
print(f"Device type: {device_type}")
print(f"Capacity: {capacity} g")
assert len(device_type) > 0
assert capacity > 0
print("PASS: device type and capacity valid")

### I1 - MT-SICS levels

In [ ]:
levels = await backend._query_mt_sics_levels()
print(f"Supported levels: {sorted(levels)}")
assert 0 in levels
assert 1 in levels
print("PASS: levels 0 and 1 present (always required)")

### S - Stable weight (ensure pan is empty)

In [ ]:
weight = await backend.read_stable_weight()
print(f"Stable weight (empty pan): {weight} g")
assert isinstance(weight, float)
print("PASS: stable weight returns float")

### SI - Weight immediately

In [ ]:
weight = await backend.read_weight_value_immediately()
print(f"Immediate weight: {weight} g")
assert isinstance(weight, float)
print("PASS: immediate weight returns float")

### Z - Zero (stable)

In [ ]:
await scale.zero(timeout="stable")
weight = await backend.read_weight_value_immediately()
print(f"Weight after zero: {weight} g")
assert abs(weight) < 0.001, f"Expected ~0 after zero, got {weight}"
print("PASS: zero sets weight to ~0")

### ZI - Zero immediately

In [ ]:
await scale.zero(timeout=0)
weight = await backend.read_weight_value_immediately()
print(f"Weight after zero immediately: {weight} g")
assert abs(weight) < 0.01
print("PASS: zero immediately sets weight to ~0")

---
## 3. Level 1 Commands (Elementary)

**Place a known weight on the scale for tare tests.**

### T - Tare (stable)

In [ ]:
input("Place a container on the scale and press Enter...")
await scale.zero(timeout="stable")
await scale.tare(timeout="stable")
tare = await scale.request_tare_weight()
print(f"Tare weight: {tare} g")
assert tare > 0, f"Expected positive tare, got {tare}"
print("PASS: tare stores container weight")

### TA - Request tare weight

In [ ]:
tare = await scale.request_tare_weight()
print(f"Tare weight from memory: {tare} g")
assert isinstance(tare, float)
print("PASS: tare weight readable from memory")

### TAC - Clear tare

In [ ]:
await backend.clear_tare()
tare_after_clear = await scale.request_tare_weight()
print(f"Tare after clear: {tare_after_clear} g")
assert abs(tare_after_clear) < 0.001, f"Expected ~0 after clear, got {tare_after_clear}"
print("PASS: clear tare resets tare to 0")

### SC - Dynamic weight (timed read)

In [ ]:
weight = await backend.read_dynamic_weight(timeout=3.0)
print(f"Dynamic weight (3s timeout): {weight} g")
assert isinstance(weight, float)
print("PASS: dynamic weight returns float")

### C - Cancel all

In [ ]:
await backend.cancel_all()
print("PASS: cancel_all completed without error")

### D / DW - Display text

In [ ]:
await backend.set_display_text("PLR TEST")
import asyncio
await asyncio.sleep(2)
await backend.set_weight_display()
print("PASS: display text set and restored (check terminal if connected)")

### ZC / TC - Timed zero and tare

These are known to return ES (syntax error) on the WXS205SDU despite being in the spec.

In [ ]:
from pylabrobot.scales.mettler_toledo.errors import MettlerToledoError

for cmd_name, cmd in [("ZC", "ZC 5000"), ("TC", "TC 5000")]:
    try:
        await backend.send_command(cmd)
        print(f"{cmd_name}: supported on this device")
    except MettlerToledoError as e:
        if "Syntax error" in str(e):
            print(f"{cmd_name}: not supported (ES) - expected for WXS205SDU")
        else:
            print(f"{cmd_name}: unexpected error: {e}")

---
## 4. Level 2 Commands (Extended)

### M21 - Set host unit to grams

In [ ]:
if 2 in backend._mt_sics_levels:
    await backend.set_host_unit_grams()
    print("PASS: host unit set to grams")
else:
    print("SKIP: Level 2 not supported")

### I50 - Remaining weighing range (multi-response)

In [ ]:
if 2 in backend._mt_sics_levels:
    remaining = await backend.request_remaining_weighing_range()
    print(f"Remaining weighing range: {remaining} g")
    assert remaining > 0
    assert remaining <= backend.capacity
    print("PASS: remaining range is positive and within capacity")
else:
    print("SKIP: Level 2 not supported")

### M28 - Temperature sensor

In [ ]:
if 2 in backend._mt_sics_levels:
    temp = await backend.measure_temperature()
    print(f"Scale temperature: {temp} C")
    assert 5 < temp < 50, f"Temperature {temp} C outside reasonable range"
    print("PASS: temperature sensor returns reasonable value")
else:
    print("SKIP: Level 2 not supported")

---
## 5. Frontend Integration

Test the Scale frontend methods that delegate to the backend.

In [ ]:
# Zero via frontend
await scale.zero(timeout="stable")
w = await scale.read_weight(timeout=0)
print(f"Frontend zero + read: {w} g")
assert abs(w) < 0.001

# Tare via frontend
input("Place a container on the scale and press Enter...")
await scale.tare(timeout="stable")
tare = await scale.request_tare_weight()
print(f"Frontend tare weight: {tare} g")
assert tare > 0

# Read via frontend
w = await scale.read_weight(timeout="stable")
print(f"Frontend read (should be ~0 with container tared): {w} g")
assert abs(w) < 0.1

print("PASS: all frontend methods delegate correctly")

---
## 6. Performance

In [ ]:
import time
import numpy as np

# Stable read latency
times_stable = []
for _ in range(10):
    t0 = time.monotonic_ns()
    await scale.read_weight(timeout="stable")
    t1 = time.monotonic_ns()
    times_stable.append((t1 - t0) / 1e6)

# Immediate read latency
times_immediate = []
for _ in range(10):
    t0 = time.monotonic_ns()
    await scale.read_weight(timeout=0)
    t1 = time.monotonic_ns()
    times_immediate.append((t1 - t0) / 1e6)

print(f"Stable read:    {np.mean(times_stable):.2f} +/- {np.std(times_stable):.2f} ms")
print(f"Immediate read: {np.mean(times_immediate):.2f} +/- {np.std(times_immediate):.2f} ms")

---
## 7. Teardown

In [ ]:
plr_logger.info("=== Hardware validation ended ===")
await scale.stop()
print(f"Log file: {log_file}")